# CMIP6 XHWI Monthly Accumulated Index - PyTorch/GPU Experimental

This notebook is an experimental GPU-oriented implementation of the CMIP6 XHWI workflow.

It keeps the same scientific logic as the ERA5 workflow and the reference CMIP6 notebook:

- use daily `tasmax` from 1961-1990 as calibration;
- build one empirical CDF per calendar month and grid cell;
- apply each monthly CDF only to the corresponding calendar month;
- compute hourly XHWI from hourly `tas` and derived `hurs`;
- aggregate hourly XHWI into daily products and monthly accumulated values;
- write only the final monthly accumulated NetCDF file per scenario.

The implementation differs computationally: it processes one `scenario x calendar month x spatial block` at a time and moves each block to PyTorch. This is intended to reduce RAM pressure and use the Colab GPU for the most expensive numerical steps.

Each section is organized as a future module/script boundary.


## Future File: `cmip6/spatial/notebooks/setup_colab.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

import os

user = input('Who is? ')
target_dir = 'drive/My Drive/Mestrado/lammoc/indices'

if user.upper() in {'LIVIA', 'VITOR'}:
    if os.getcwd() == '/content':
        os.chdir(target_dir)
    print(os.getcwd())
    print(os.listdir())
else:
    raise ValueError("Expected user to be 'LIVIA' or 'VITOR'.")


## Future File: `cmip6/spatial/notebooks/install_dependencies.py`

In [ ]:
!pip -q install zarr dask distributed numcodecs cftime netCDF4 h5netcdf scipy
!pip -q install xclim climate-indices


## Future File: `cmip6/spatial/scripts/src/config/settings.py`

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
MODEL_ID = 'BCC-CSM2-MR'
GRID_LABEL = 'gn'
MEMBER_ID = 'r1i1p1f1'
CALIBRATION_PERIOD = ('1961-01-01', '1990-12-31')

if (BASE_DIR / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR
elif (BASE_DIR / 'cmip6' / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR / 'cmip6'
elif (BASE_DIR / 'index-xhwi' / 'cmip6' / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR / 'index-xhwi' / 'cmip6'
else:
    raise FileNotFoundError(f'Could not locate CMIP6 root from {BASE_DIR}')

MODEL_ROOT = CMIP6_ROOT / MODEL_ID
OUTPUT_DIR = MODEL_ROOT / 'results' / 'xhwi_torch'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = {
    'historical': {
        'tas': MODEL_ROOT / 'historical' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'historical' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'tasmax': MODEL_ROOT / 'historical' / 'day' / 'tasmax' / GRID_LABEL / 'ensemble_mean.zarr',
    },
    'ssp245': {
        'tas': MODEL_ROOT / 'ssp245' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'ssp245' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'tasmax': MODEL_ROOT / 'ssp245' / 'day' / 'tasmax' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
    },
    'ssp585': {
        'tas': MODEL_ROOT / 'ssp585' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'ssp585' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'tasmax': MODEL_ROOT / 'ssp585' / 'day' / 'tasmax' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
    },
}

TEMPERATURE_THRESHOLD_C = 32.0
CDF_THRESHOLD_PERCENT = 95.0
STANDARD_PRESSURE_PA = 101325.0

# Tune these values in Colab depending on available GPU RAM.
LAT_BLOCK_SIZE = 16
LON_BLOCK_SIZE = 16
TIME_CHUNK_HOURLY = 24 * 31
SPATIAL_CHUNK = 32
TORCH_DTYPE = 'float32'

print(f'CMIP6 root: {CMIP6_ROOT}')
print(f'Model root: {MODEL_ROOT}')
for scenario, paths in SCENARIOS.items():
    for name, path in paths.items():
        print(f'{scenario:10s} {name:6s}: {path}')


## Future File: `cmip6/spatial/scripts/src/utils/imports.py`

In [ ]:
from datetime import datetime, timezone

import gc
import numpy as np
import xarray as xr
import torch
from dask.diagnostics import ProgressBar

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32 if TORCH_DTYPE == 'float32' else torch.float64

print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))


## Future File: `cmip6/spatial/scripts/src/preprocessing/cmip6.py`

In [ ]:
def clean_cmip6_dims(ds: xr.Dataset) -> xr.Dataset:
    """Keep only time, lat, and lon dimensions and sort them."""
    keep_dims = {'time', 'lat', 'lon'}

    vars_to_keep = [
        var for var in ds.data_vars
        if set(ds[var].dims).issubset(keep_dims)
    ]
    ds = ds[vars_to_keep]

    coords_to_drop = [coord for coord in ds.coords if coord not in keep_dims]
    ds = ds.drop_vars(coords_to_drop, errors='ignore')

    for dim in ['time', 'lat', 'lon']:
        if dim in ds.dims:
            ds = ds.sortby(dim)

    return ds


def open_clean_zarr(path: Path, chunks: dict | None = None) -> xr.Dataset:
    ds = xr.open_zarr(path, chunks=chunks)
    return clean_cmip6_dims(ds)


def interpolate_to_hourly(ds: xr.Dataset) -> xr.Dataset:
    return ds.sortby('time').resample(time='1h').interpolate('linear')


def kelvin_to_celsius(da: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get('units', '')).lower()
    out = da - 273.15 if units in {'k', 'kelvin'} else da
    out = out.copy()
    out.attrs.update(da.attrs)
    out.attrs['units'] = 'degC'
    return out


## Future File: `cmip6/spatial/scripts/src/features/humidity.py`

In [ ]:
def specific_to_relative_humidity_standard_pressure(
    huss: xr.DataArray,
    tas: xr.DataArray,
    p0: float = STANDARD_PRESSURE_PA,
    clip: bool = True,
) -> xr.DataArray:
    """Convert specific humidity to relative humidity using standard pressure."""
    epsilon = 0.622
    tas_c = tas - 273.15 if str(tas.attrs.get('units', '')).lower() in {'k', 'kelvin'} else tas

    e = (huss * p0) / (epsilon + (1 - epsilon) * huss)
    es = 611.2 * np.exp((17.67 * tas_c) / (tas_c + 243.5))
    hurs = 100.0 * e / es

    if clip:
        hurs = hurs.clip(min=0, max=100)

    hurs.name = 'hurs'
    hurs.attrs.update({
        'standard_name': 'relative_humidity',
        'long_name': 'Relative humidity',
        'units': '%',
        'description': (
            'Relative humidity computed from specific humidity and air temperature '
            'assuming standard surface pressure p = 101325 Pa. Saturation vapor '
            'pressure computed using Bolton 1980.'
        ),
        'assumed_pressure': '101325 Pa',
    })
    return hurs


## Future File: `cmip6/spatial/scripts/src/torch_ops/cdf.py`

In [ ]:
def torch_match_cdf_linear(
    tas_hourly_c: torch.Tensor,
    tasmax_calibration_c: torch.Tensor,
) -> torch.Tensor:
    """Match hourly tas to empirical calibration CDF using linear interpolation."""
    time_size, y_size, x_size = tas_hourly_c.shape

    values = tas_hourly_c.reshape(time_size, -1).transpose(0, 1).contiguous()
    calibration = tasmax_calibration_c.reshape(tasmax_calibration_c.shape[0], -1).transpose(0, 1).contiguous()

    finite_cal = torch.isfinite(calibration)
    n_valid = finite_cal.sum(dim=1)
    calibration_sorted = calibration.masked_fill(~finite_cal, float('inf')).sort(dim=1).values

    finite_values = torch.isfinite(values)
    safe_values = values.masked_fill(~finite_values, 0.0)

    idx_right = torch.searchsorted(calibration_sorted, safe_values, right=False)
    max_idx = torch.clamp(n_valid - 1, min=0).unsqueeze(1)
    idx1 = torch.minimum(idx_right, max_idx).long()
    idx0 = torch.clamp(idx1 - 1, min=0).long()

    x0 = calibration_sorted.gather(1, idx0)
    x1 = calibration_sorted.gather(1, idx1)

    n_valid_f = n_valid.clamp(min=1).unsqueeze(1).to(values.dtype)
    y0 = (idx0.to(values.dtype) + 1.0) / n_valid_f
    y1 = (idx1.to(values.dtype) + 1.0) / n_valid_f

    denom = x1 - x0
    frac = torch.where(torch.abs(denom) > 0, (safe_values - x0) / denom, torch.zeros_like(safe_values))
    target = y0 + frac * (y1 - y0)

    first = calibration_sorted[:, 0].unsqueeze(1)
    last = calibration_sorted.gather(1, max_idx.long())
    target = torch.where(safe_values < first, torch.zeros_like(target), target)
    target = torch.where(safe_values >= last, torch.ones_like(target), target)
    target = torch.where((n_valid < 2).unsqueeze(1), torch.full_like(target, float('nan')), target)
    target = torch.where(finite_values, target, torch.full_like(target, float('nan')))
    target = torch.clamp(target, min=0.0, max=1.0)

    return target.transpose(0, 1).reshape(time_size, y_size, x_size)


## Future File: `cmip6/spatial/scripts/src/torch_ops/xhwi.py`

In [ ]:
def torch_heatwave_index(
    tas_c: torch.Tensor,
    hurs: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    target100 = target * 100.0
    tpe = torch.clamp(target100 - CDF_THRESHOLD_PERCENT, min=0.0)
    coef = (torch.exp(tpe) * hurs) / 1000.0
    xhwi = (coef - 0.001) / 14.84

    xhwi = torch.where(tpe > 0, xhwi, torch.zeros_like(xhwi))
    xhwi = torch.where(tas_c > TEMPERATURE_THRESHOLD_C, xhwi, torch.zeros_like(xhwi))
    xhwi = torch.where(xhwi > 0.001, xhwi, torch.zeros_like(xhwi))
    return xhwi


## Future File: `cmip6/spatial/scripts/src/torch_ops/aggregations.py`

In [ ]:
def month_keys_from_time(time_coord: xr.DataArray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    years = time_coord.dt.year.values.astype(np.int64)
    months = time_coord.dt.month.values.astype(np.int64)
    days = time_coord.dt.day.values.astype(np.int64)
    day_keys = years * 10000 + months * 100 + days
    month_keys = years * 100 + months
    return day_keys, month_keys, np.asarray(time_coord.values)


def torch_monthly_accumulated_xhwi(
    xhwi: torch.Tensor,
    time_coord: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray]:
    """Aggregate hourly XHWI into monthly accumulated values on GPU."""
    day_keys, month_keys_by_time, time_values = month_keys_from_time(time_coord)
    unique_days = np.unique(day_keys)

    daily_values = []
    daily_month_keys = []

    for day_key in unique_days:
        idx_np = np.flatnonzero(day_keys == day_key)
        idx = torch.as_tensor(idx_np, device=xhwi.device, dtype=torch.long)
        xhwi_day = xhwi.index_select(0, idx)
        active_hours = (xhwi_day != 0).sum(dim=0).to(xhwi.dtype)
        daily_sum = xhwi_day.sum(dim=0)
        daily_values.append(active_hours * daily_sum)
        daily_month_keys.append(month_keys_by_time[idx_np[0]])

    if not daily_values:
        raise ValueError('No daily values were generated for this block.')

    daily_stack = torch.stack(daily_values, dim=0)
    daily_month_keys = np.asarray(daily_month_keys)
    unique_months = np.unique(daily_month_keys)

    monthly_values = []
    monthly_time_values = []
    for month_key in unique_months:
        day_idx_np = np.flatnonzero(daily_month_keys == month_key)
        day_idx = torch.as_tensor(day_idx_np, device=xhwi.device, dtype=torch.long)
        monthly_values.append(daily_stack.index_select(0, day_idx).sum(dim=0))
        first_time_idx = np.flatnonzero(month_keys_by_time == month_key)[0]
        monthly_time_values.append(time_values[first_time_idx])

    monthly = torch.stack(monthly_values, dim=0).detach().cpu().numpy().astype('float32')
    return monthly, np.asarray(monthly_time_values)


## Future File: `cmip6/spatial/scripts/src/io/writers.py`

In [ ]:
def build_monthly_output_dataset(monthly: xr.DataArray, scenario: str) -> xr.Dataset:
    ds = monthly.to_dataset(name='xhwi_monthly_accumulated')

    if 'lat' in ds.coords:
        ds['lat'].attrs.update({'standard_name': 'latitude', 'long_name': 'Latitude', 'units': 'degrees_north', 'axis': 'Y'})
    if 'lon' in ds.coords:
        ds['lon'].attrs.update({'standard_name': 'longitude', 'long_name': 'Longitude', 'units': 'degrees_east', 'axis': 'X'})
    if 'time' in ds.coords:
        ds['time'].attrs.update({'standard_name': 'time', 'long_name': 'Time', 'axis': 'T'})

    ds['xhwi_monthly_accumulated'].attrs.update({
        'long_name': 'Monthly accumulated Extreme Heatwave Index',
        'units': '1',
        'cell_methods': 'time: sum',
        'description': (
            'Monthly sum of daily XHWI products. Each daily product is the number '
            'of hours with nonzero XHWI multiplied by the daily sum of hourly XHWI.'
        ),
    })

    ds.attrs.update({
        'Conventions': 'CF-1.10',
        'title': f'Monthly accumulated XHWI for CMIP6 {MODEL_ID} {scenario}',
        'source': (
            f'CMIP6 model {MODEL_ID}, experiment {scenario}, member {MEMBER_ID}; '
            'XHWI computed from 3-hourly tas and huss interpolated to hourly resolution; '
            'calendar-month-specific calibration CDF computed from daily tasmax for 1961-1990. '
            'Experimental PyTorch blockwise implementation.'
        ),
        'creator': 'LAMMOC-UFF',
        'contact': 'mcataldi@id.uff.br',
        'institution': 'Laboratory for Monitoring and Modeling of Climate Systems, Federal Fluminense University',
        'creation_date': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'model_id': MODEL_ID,
        'experiment_id': scenario,
        'member_id': MEMBER_ID,
        'grid_label': GRID_LABEL,
        'calibration_period': f'{CALIBRATION_PERIOD[0]} to {CALIBRATION_PERIOD[1]}',
        'calibration_method': 'Separate empirical CDF for each calendar month and grid cell.',
        'compute_backend': f'PyTorch on {DEVICE.type}',
        'input_variables': 'tas, huss, tasmax',
        'humidity_method': 'Specific humidity converted to relative humidity assuming p = 101325 Pa.',
        'temperature_threshold': f'{TEMPERATURE_THRESHOLD_C} degC',
        'cdf_threshold': f'p{int(CDF_THRESHOLD_PERCENT)}',
    })
    return ds


def write_monthly_netcdf(ds: xr.Dataset, scenario: str) -> Path:
    output_path = OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_{scenario}_{MEMBER_ID}_monthly_accumulated_torch.nc'
    ds = ds.sortby('time')

    if 'time' in ds.indexes and not ds.indexes['time'].is_monotonic_increasing:
        raise ValueError('Output time coordinate is not monotonic increasing.')
    if 'time' in ds.indexes and not ds.indexes['time'].is_unique:
        raise ValueError('Output time coordinate contains duplicated values.')

    encoding = {
        'xhwi_monthly_accumulated': {
            'zlib': True,
            'complevel': 4,
            '_FillValue': np.float32(np.nan),
            'dtype': 'float32',
        }
    }

    with ProgressBar():
        ds.to_netcdf(output_path, engine='netcdf4', encoding=encoding)

    return output_path


## Future File: `cmip6/spatial/scripts/src/pipeline/data_access.py`

In [ ]:
def open_calibration_tasmax() -> xr.DataArray:
    tasmax_ds = open_clean_zarr(
        SCENARIOS['historical']['tasmax'],
        chunks={'time': -1, 'lat': SPATIAL_CHUNK, 'lon': SPATIAL_CHUNK},
    )
    tasmax = kelvin_to_celsius(tasmax_ds['tasmax'])
    tasmax = tasmax.sel(time=slice(*CALIBRATION_PERIOD))

    if tasmax.sizes.get('time', 0) == 0:
        raise ValueError(f'No tasmax data found for calibration period {CALIBRATION_PERIOD}.')

    return tasmax.rename({'time': 'calibration_time'})


def open_hourly_scenario_inputs(scenario: str) -> tuple[xr.DataArray, xr.DataArray]:
    paths = SCENARIOS[scenario]
    chunks = {'time': TIME_CHUNK_HOURLY, 'lat': SPATIAL_CHUNK, 'lon': SPATIAL_CHUNK}

    tas_ds = interpolate_to_hourly(open_clean_zarr(paths['tas'], chunks=chunks))
    huss_ds = interpolate_to_hourly(open_clean_zarr(paths['huss'], chunks=chunks))

    tas = tas_ds['tas']
    huss = huss_ds['huss']

    tas_c = kelvin_to_celsius(tas)
    hurs = specific_to_relative_humidity_standard_pressure(huss=huss, tas=tas)

    return tas_c, hurs


def iter_spatial_blocks(lat_size: int, lon_size: int, lat_block: int, lon_block: int):
    for lat_start in range(0, lat_size, lat_block):
        lat_stop = min(lat_start + lat_block, lat_size)
        for lon_start in range(0, lon_size, lon_block):
            lon_stop = min(lon_start + lon_block, lon_size)
            yield slice(lat_start, lat_stop), slice(lon_start, lon_stop)


## Future File: `cmip6/spatial/scripts/src/pipeline/block_processor.py`

In [ ]:
def load_block_np(da: xr.DataArray, lat_slice: slice, lon_slice: slice) -> np.ndarray:
    return da.isel(lat=lat_slice, lon=lon_slice).load().values.astype('float32')


def process_month_block_torch(
    tas_c_month: xr.DataArray,
    hurs_month: xr.DataArray,
    tasmax_calibration_month: xr.DataArray,
    lat_slice: slice,
    lon_slice: slice,
) -> tuple[np.ndarray, np.ndarray]:
    tas_np = load_block_np(tas_c_month, lat_slice, lon_slice)
    hurs_np = load_block_np(hurs_month, lat_slice, lon_slice)
    tasmax_np = load_block_np(tasmax_calibration_month, lat_slice, lon_slice)

    tas_t = torch.as_tensor(tas_np, dtype=DTYPE, device=DEVICE)
    hurs_t = torch.as_tensor(hurs_np, dtype=DTYPE, device=DEVICE)
    tasmax_t = torch.as_tensor(tasmax_np, dtype=DTYPE, device=DEVICE)

    target_t = torch_match_cdf_linear(tas_t, tasmax_t)
    xhwi_t = torch_heatwave_index(tas_c=tas_t, hurs=hurs_t, target=target_t)
    monthly_np, monthly_time = torch_monthly_accumulated_xhwi(xhwi_t, tas_c_month['time'])

    del tas_t, hurs_t, tasmax_t, target_t, xhwi_t
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return monthly_np, monthly_time


## Future File: `cmip6/spatial/scripts/src/pipeline/monthly_pipeline.py`

In [ ]:
def compute_scenario_monthly_xhwi_torch(scenario: str) -> xr.Dataset:
    print(f'Opening and preprocessing {scenario} inputs...')
    tas_c, hurs = open_hourly_scenario_inputs(scenario)
    tasmax_calibration = open_calibration_tasmax()

    lat_values = tas_c['lat'].values
    lon_values = tas_c['lon'].values
    lat_size = tas_c.sizes['lat']
    lon_size = tas_c.sizes['lon']

    scenario_month_arrays = []
    scenario_month_times = []

    for month in range(1, 13):
        print(f'Processing {scenario}, calendar month {month:02d}...')
        tas_c_month = tas_c.sel(time=tas_c['time.month'] == month)
        hurs_month = hurs.sel(time=hurs['time.month'] == month)
        tasmax_calibration_month = tasmax_calibration.sel(
            calibration_time=tasmax_calibration['calibration_time.month'] == month
        )

        if tas_c_month.sizes.get('time', 0) == 0:
            print(f'No hourly tas data found for {scenario}, month {month:02d}; skipping.')
            continue
        if tasmax_calibration_month.sizes.get('calibration_time', 0) == 0:
            raise ValueError(f'No calibration tasmax data found for month {month}.')

        month_template = None
        month_time = None

        for lat_slice, lon_slice in iter_spatial_blocks(lat_size, lon_size, LAT_BLOCK_SIZE, LON_BLOCK_SIZE):
            print(
                f'  block lat[{lat_slice.start}:{lat_slice.stop}] '
                f'lon[{lon_slice.start}:{lon_slice.stop}]'
            )
            block_np, block_time = process_month_block_torch(
                tas_c_month=tas_c_month,
                hurs_month=hurs_month,
                tasmax_calibration_month=tasmax_calibration_month,
                lat_slice=lat_slice,
                lon_slice=lon_slice,
            )

            if month_template is None:
                month_time = block_time
                month_template = np.full((len(month_time), lat_size, lon_size), np.nan, dtype='float32')
            elif len(block_time) != len(month_time):
                raise ValueError('Inconsistent monthly time length across spatial blocks.')

            month_template[:, lat_slice, lon_slice] = block_np

        if month_template is not None:
            scenario_month_arrays.append(month_template)
            scenario_month_times.append(month_time)

    if not scenario_month_arrays:
        raise ValueError(f'No monthly XHWI outputs were generated for {scenario}.')

    monthly_values = np.concatenate(scenario_month_arrays, axis=0)
    monthly_time = np.concatenate(scenario_month_times, axis=0)

    monthly = xr.DataArray(
        monthly_values,
        dims=('time', 'lat', 'lon'),
        coords={'time': monthly_time, 'lat': lat_values, 'lon': lon_values},
        name='xhwi_monthly_accumulated',
    ).sortby('time')

    return build_monthly_output_dataset(monthly, scenario=scenario)


## Run: Historical

In [ ]:
ds_historical_monthly = compute_scenario_monthly_xhwi_torch('historical')
display(ds_historical_monthly)
historical_output = write_monthly_netcdf(ds_historical_monthly, 'historical')
print(historical_output)


## Run: SSP2-4.5

In [ ]:
ds_ssp245_monthly = compute_scenario_monthly_xhwi_torch('ssp245')
display(ds_ssp245_monthly)
ssp245_output = write_monthly_netcdf(ds_ssp245_monthly, 'ssp245')
print(ssp245_output)


## Run: SSP5-8.5

In [ ]:
ds_ssp585_monthly = compute_scenario_monthly_xhwi_torch('ssp585')
display(ds_ssp585_monthly)
ssp585_output = write_monthly_netcdf(ds_ssp585_monthly, 'ssp585')
print(ssp585_output)


## Optional: Small Numerical Smoke Test

Use this cell only after the functions above are loaded. It runs a tiny block for one scenario and one month to verify shapes before running a full scenario.


In [ ]:
# Optional smoke test. Uncomment to run.
# scenario = 'historical'
# month = 1
# tas_c, hurs = open_hourly_scenario_inputs(scenario)
# tasmax_calibration = open_calibration_tasmax()
# tas_c_month = tas_c.sel(time=tas_c['time.month'] == month)
# hurs_month = hurs.sel(time=hurs['time.month'] == month)
# tasmax_month = tasmax_calibration.sel(calibration_time=tasmax_calibration['calibration_time.month'] == month)
# block_np, block_time = process_month_block_torch(
#     tas_c_month=tas_c_month,
#     hurs_month=hurs_month,
#     tasmax_calibration_month=tasmax_month,
#     lat_slice=slice(0, min(4, tas_c.sizes['lat'])),
#     lon_slice=slice(0, min(4, tas_c.sizes['lon'])),
# )
# print(block_np.shape, block_time[:3])


## Optional CF Metadata Inspection

In [ ]:
for path in [
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_historical_{MEMBER_ID}_monthly_accumulated_torch.nc',
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_ssp245_{MEMBER_ID}_monthly_accumulated_torch.nc',
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_ssp585_{MEMBER_ID}_monthly_accumulated_torch.nc',
]:
    if path.exists():
        ds = xr.open_dataset(path)
        print(path)
        print(ds)
        print(ds.attrs)
        print(ds['xhwi_monthly_accumulated'].attrs)
        ds.close()
